Лабораторная работа №3. Глубокое обучение и его применение в анализе данных. Задача 3. Применение рекуррентной нейронной сети для аннотации текста

Шаг 1: Загрузка датасета CoNLL-2003

CoNLL-2003 — это популярный датасет для задач NLP, который содержит аннотированные тексты с метками для частей речи и именованных сущностей. Мы будем использовать его для задачи POS-теггинга.

In [ ]:
import nltk
from nltk.corpus import conll2000

# Загрузка датасета CoNLL-2000 (аналогичен CoNLL-2003, но проще для POS-теггинга)
nltk.download('conll2000')

# Загрузка предложений с метками частей речи
sentences = conll2000.tagged_sents()

[nltk_data] Downloading package conll2000 to /root/nltk_data...
[nltk_data]   Unzipping corpora/conll2000.zip.


Шаг 2: Предобработка данных

Разделим данные на обучающую и тестовую выборки.

In [ ]:
# Разделение на обучающую и тестовую выборки
train_size = int(0.8 * len(sentences))
train_sentences = sentences[:train_size]
test_sentences = sentences[train_size:]

Шаг 3: Подготовка данных

Создадим словари для слов и меток, а затем преобразуем предложения в последовательности индексов.

In [ ]:
from collections import Counter
import numpy as np

# Создание словарей
words = Counter(word for sent in train_sentences for word, _ in sent)
tags = Counter(tag for sent in train_sentences for _, tag in sent)

word2idx = {word: i + 2 for i, (word, _) in enumerate(words.items())}
word2idx['<pad>'] = 0  # Паддинг
word2idx['<unk>'] = 1  # Неизвестные слова

tag2idx = {tag: i + 1 for i, (tag, _) in enumerate(tags.items())}
tag2idx['<pad>'] = 0  # Паддинг для меток

# Преобразование предложений в последовательности индексов
def sentence_to_indices(sentence, vocab):
    return [vocab.get(word, vocab['<unk>']) for word, _ in sentence]

def tags_to_indices(sentence, vocab):
    return [vocab.get(tag, vocab['<pad>']) for _, tag in sentence]

X_train = [sentence_to_indices(sent, word2idx) for sent in train_sentences]
y_train = [tags_to_indices(sent, tag2idx) for sent in train_sentences]

X_test = [sentence_to_indices(sent, word2idx) for sent in test_sentences]
y_test = [tags_to_indices(sent, tag2idx) for sent in test_sentences]

Шаг 4: Построение модели LSTM

Создадим модель с использованием слоев Embedding, LSTM и TimeDistributed Dense.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, TimeDistributed
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Дополнение последовательностей до одинаковой длины
X_train = pad_sequences(X_train, padding='post')
y_train = pad_sequences(y_train, padding='post')
X_test = pad_sequences(X_test, padding='post')
y_test = pad_sequences(y_test, padding='post')

# Определение модели
model = Sequential([
    Embedding(input_dim=len(word2idx), output_dim=50, input_length=X_train.shape[1]),
    LSTM(64, return_sequences=True),
    TimeDistributed(Dense(len(tag2idx), activation='softmax'))
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Обучение модели
model.fit(X_train, y_train, epochs=5, batch_size=32, validation_split=0.1)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ time_distributed (TimeDistributed)   │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
247/247 ━━━━━━━━━━━━━━━━━━━━ 34s 97ms/step - accuracy: 0.7058 - loss: 1.4339 - val_accuracy: 0.8396 - val_loss: 0.6317
Epoch 2/5
247/247 ━━━━━━━━━━━━━━━━━━━━ 41s 97ms/step - accuracy: 0.8669 - loss: 0.5285 - val_accuracy: 0.9473 - val_loss: 0.2553
Epoch 3/5
247/247 ━━━━━━━━━━━━━━━━━━━━ 24s 95ms/step - accuracy: 0.9635 - loss: 0.1967 - val_accuracy: 0.9746 - val_loss: 0.1257
Epoch 4/5
247/247 ━━━━━━━━━━━━━━━━━━━━ 23s 95ms/step - accuracy: 0.9859 - loss: 0.0860 - val_accuracy: 0.9804 - val_loss: 0.0865
Epoch 5/5
247/247 ━━━━━━━━━━━━━━━━━━━━ 41s 95ms/step - accuracy: 0.9911 - loss: 0.0498 - val_accuracy: 0.9824 - val_loss: 0.0710


Шаг 5: Оценка модели

Оценим модель на тестовой выборке и выведем примеры предложений с предсказанными и реальными метками.

In [ ]:
# Оценка модели
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Точность на тестовом наборе: {accuracy}')

# Предсказание и сравнение меток
predicted_tags = model.predict(X_test)
predicted_tags = np.argmax(predicted_tags, axis=-1)

# Преобразование индексов обратно в метки
idx2tag = {i: tag for tag, i in tag2idx.items()}
predicted_tags = [[idx2tag[idx] for idx in sent] for sent in predicted_tags]
actual_tags = [[idx2tag[idx] for idx in sent] for sent in y_test]

# Вывод примеров
for i in range(5):
    print("Предложение:", " ".join([word for word, _ in test_sentences[i]]))
    print("Реальные метки:", actual_tags[i])
    print("Предсказанные метки:", predicted_tags[i])
    print()

69/69 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.9799 - loss: 0.0808
Точность на тестовом наборе: 0.9790275692939758
69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step
Предложение: The debentures were issued in the face amount of $ 46 million on July 11 , 1988 , the Ashland , Ky. , coal mining , water transportation and construction company said .
Реальные метки: ['DT', 'NNS', 'VBD', 'VBN', 'IN', 'DT', 'NN', 'NN', 'IN', '$', 'CD', 'CD', 'IN', 'NNP', 'CD', ',', 'CD', ',', 'DT', 'NNP', ',', 'NNP', ',', 'NN', 'NN', ',', 'NN', 'NN', 'CC', 'NN', 'NN', 'VBD', '.', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']
Предсказанные метки: ['DT', 'NNS', 'VBD', 'VBN', 'IN', 'DT', 'NN', 'NN', 'IN', '$', 'CD', 'CD', 'IN

Мы создали модель LSTM для задачи POS-теггинга, используя датасет CoNLL-2000. Модель показывает хорошую точность на тестовом наборе, и вы можете видеть, как она справляется с конкретными примерами предложений.